In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
CSV_PATH = 'EduPredict.csv'
MODEL_PATH = 'edupredict_model.pkl'
SCALER_PATH = 'scaler.pkl'
FEATURES = ['attendance_percentage', 'lms_logins_last_30_days', 'previous_gpa', 'midterm_score']

In [ ]:
data = pd.read_csv(CSV_PATH)
data.head()

In [ ]:
data.info()
data.describe(include='all').T

In [ ]:
# Accept prior EduPredict exports that used the older names.
aliases = {'attendance_rate': 'attendance_percentage', 'lms_activity_score': 'lms_logins_last_30_days', 'gpa': 'previous_gpa'}
data = data.rename(columns={old: new for old, new in aliases.items() if old in data.columns and new not in data.columns})
for column in FEATURES:
    if column not in data.columns:
        data[column] = 0.0
    data[column] = pd.to_numeric(data[column], errors='coerce').fillna(0.0)
data[FEATURES].head()

In [ ]:
# Prefer a final outcome column; otherwise use the available midterm score.
target_column = next((column for column in ['final_score', 'performance_score', 'final_grade'] if column in data.columns), 'midterm_score')
target = pd.to_numeric(data[target_column], errors='coerce').fillna(0.0)
X = data[FEATURES]
print('Target:', target_column)
print('Rows:', len(data), '| distinct target values:', target.nunique())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
model = RandomForestRegressor(n_estimators=150, random_state=42)
model.fit(X_train_scaled, y_train)

In [ ]:
predictions = model.predict(X_test_scaled)
print('MAE:', round(mean_absolute_error(y_test, predictions), 3))
print('R²:', round(r2_score(y_test, predictions), 3))
pd.DataFrame({'actual': y_test.to_numpy(), 'predicted': predictions}).head()

In [ ]:
# Refit on all available data before saving artifacts used by the Flask application.
scaler = StandardScaler().fit(X)
model = RandomForestRegressor(n_estimators=150, random_state=42)
model.fit(scaler.transform(X), target)
with open(MODEL_PATH, 'wb') as model_file:
    pickle.dump(model, model_file)
with open(SCALER_PATH, 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)
print('Saved:', MODEL_PATH, 'and', SCALER_PATH)

In [ ]:
# Optional one-student prediction check.
example_student = pd.DataFrame([[85, 20, 3.1, 72]], columns=FEATURES)
model.predict(scaler.transform(example_student))